# <font color="red"> Cuaderno 12 — TensorFlow para Procesamiento de Texto </font>

## <font color="blue"> Por Alfredo Díaz — Maestría en Ciencia de Datos </font>

---

## 📍 Posición en la secuencia de aprendizaje

```
Cuaderno 10 → Cuaderno 11 → ► Cuaderno 12 ◄
  Fundamentos    Embeddings     TensorFlow Text
  RNN/LSTM/GRU   Gensim/FastTF  Unicode · API · RNN desde cero
```

Este cuaderno **consolida y profundiza** lo aprendido en los cuadernos anteriores, enfocándose en tres pilares:

1. **TensorFlow para texto**: cómo TF maneja cadenas, Unicode y tensores de texto.
2. **Pipeline completo** de clasificación de texto con Keras (Tokenizer → Embedding → LSTM → predicción).
3. **RNN desde cero con NumPy**: implementación manual que desnuda la matemática interna de una red recurrente.

---

## 📋 Tabla de Contenidos

1. [TensorFlow y texto Unicode](#1-unicode) — `tf.string`, codificaciones, RaggedTensors
2. [API de texto de TF](#2-api) — `unicode_decode`, `unicode_encode`, `unicode_transcode`
3. [Pipeline clásico con Keras Tokenizer](#3-tokenizer) — Tokenizer → pad_sequences → Embedding → LSTM
4. [Clasificador de sentimientos ampliado](#4-sentimientos) — Dataset mayor, métricas, curvas de aprendizaje
5. [Comparativa: Tokenizer legacy vs TextVectorization moderno](#5-comparativa)
6. [RNN desde cero con NumPy](#6-numpy-rnn) — forward pass, BPTT, generación de municipios
7. [Comparación: NumPy vs Keras](#7-comparacion)
8. [Ejercicios propuestos](#8-ejercicios)

---
> **Prerrequisitos**: Cuadernos 10 y 11 — RNNs, LSTM, GRU, Embeddings con Keras


---

# 🧭 Mapa conceptual de este cuaderno

## ¿Por qué TensorFlow tiene operaciones específicas para texto?

El texto es **datos no estructurados** con características únicas que lo diferencian de imágenes o series numéricas:

| Característica del texto | Implicación técnica |
|--------------------------|---------------------|
| Longitud variable | Necesita padding o RaggedTensors |
| Caracteres multilingües | Requiere manejo Unicode |
| Vocabulario abierto | Palabras fuera del vocabulario (OOV) |
| Orden importa | Modelos secuenciales (RNN/Transformer) |
| Semántica distribuida | Embeddings en lugar de one-hot |

## El flujo completo texto → predicción en TensorFlow

```
Texto crudo (str)
    │
    ▼ tf.string / Unicode
Representación interna TF (puntos de código, bytes, RaggedTensor)
    │
    ▼ Tokenizer / TextVectorization
Secuencias de enteros  [4, 12, 7, 0, 0]
    │
    ▼ Embedding Layer
Vectores densos  [[v4], [v12], [v7], [0], [0]]
    │
    ▼ LSTM / GRU / Bidireccional
Estado oculto con contexto  h_T
    │
    ▼ Dense + activación
Predicción (clase, probabilidad, siguiente token...)
```

Cada bloque del flujo tiene sus propias herramientas en TensorFlow, y este cuaderno las recorre todas.


---

# 📝 1. TensorFlow y Texto Unicode

---

## ¿Por qué Unicode importa en NLP?

El texto en el mundo real es multilingüe. El español tiene tildes (á, é, ñ), el chino tiene miles de caracteres, el árabe se escribe de derecha a izquierda. **Unicode** es el estándar que asigna un número único a cada carácter de todos los idiomas humanos.

### Codificaciones comunes

| Codificación | Bits por carácter | Soporte | Uso típico |
|---|---|---|---|
| **ASCII** | 7 bits | 128 caracteres (inglés básico) | Código fuente antiguo |
| **Latin-1** | 8 bits | 256 caracteres (Europa occidental) | Archivos legacy |
| **UTF-8** | 8–32 bits (variable) | Todo Unicode | Web, Python 3, JSON |
| **UTF-16** | 16–32 bits | Todo Unicode | Windows, Java, JavaScript |
| **UTF-32** | 32 bits (fijo) | Todo Unicode | Procesamiento interno |

### TensorFlow y Unicode

TensorFlow trabaja internamente con cadenas como **bytes en UTF-8**. El tipo `tf.string` almacena secuencias de bytes, **no** caracteres directamente. Por eso es importante entender cómo convertir entre representaciones.

### Dos representaciones en TF

```
"Hola"  →  [72, 111, 108, 97]   (puntos de código Unicode / enteros)
        →  b'Hola'  (bytes UTF-8)
```

La representación como **vector de enteros** es la más útil para NLP: cada carácter o subpalabra se convierte en un índice que puede alimentar una capa Embedding.


In [ ]:
import tensorflow as tf
import numpy as np

##tf.string
TensorFlow tf.string dtype le permite construir tensores de cadenas de bytes. Son cadenas Unicode UTF-8 codificada por defecto.

In [ ]:
tf.constant(u"Hola alumno de la maestría 😊")

A tf.string trata tensor byte cadenas como unidades atómicas. Esto le permite almacenar cadenas de bytes de diferentes longitudes. La longitud de la cuerda no está incluida en las dimensiones del tensor.

In [ ]:
tf.constant([u"Estudiante", u"Bienvenido!"]).shape

Si utiliza Python a cadenas de construir, nota que los literales de cadena son Unicode con codificación por defecto.

### Hay dos formas estándar de representar una cadena Unicode en TensorFlow:

**string escalar** - donde la secuencia de puntos de código se codifica utilizando una conocida codificación de caracteres .
**int32 vector** - donde cada posición contiene un único punto de código.

Crea una cadena de texto en formato UTF-8 utilizando TensorFlow, específicamente una constante tf.constant. El texto "Ciencia de datos" se representa como una cadena Unicode (en formato UTF-8) y se almacena como una constante en TensorFlow.

In [ ]:
text_utf8 = tf.constant(u"Ciencia de datos")
text_utf8

Unicode string: Cadena de texto que puede contener caracteres de varios idiomas o símbolos especiales.

UTF-16-BE: Es una forma de codificación de texto que usa 16 bits por carácter. "BE" significa Big Endian, que se refiere al orden de los bytes en la memoria (el byte más significativo se almacena primero).

Encoded string scalar: Una cadena de texto que se representa como un valor escalar (un solo valor, no una secuencia o colección de valores).

In [ ]:
# Cadena Unicode, representada como un escalar de cadena codificado en UTF-16-BE.
text_utf16be = tf.constant(u"Ciencia de Datos".encode("UTF-16-BE"))
text_utf16be

u"Ciencia de datos": Es una cadena Unicode que contiene el texto "Ciencia de datos". El prefijo u asegura que estamos utilizando una cadena Unicode.

[ord(char) for char in u"Ciencia de datos"]: Esta es una comprensión de lista que itera sobre cada carácter de la cadena "Ciencia de datos". La función ord(char) convierte cada carácter en su punto de código Unicode, que es un número entero único que representa ese carácter.

Por ejemplo, para el carácter 'C', ord('C') devuelve el valor 67, que es el punto de código Unicode para la letra "C".

tf.constant(...): Convierte la lista de puntos de código Unicode en una constante de TensorFlow. Esta constante será un tensor que contiene los valores de los puntos de código Unicode de cada carácter en la cadena.

In [ ]:

# Cadena Unicode, representada como un vector de puntos de código Unicode.
text_chars = tf.constant([ord(char) for char in u"Ciencia de datos"])
text_chars


TensorFlow proporciona varias funciones útiles para convertir entre representaciones de texto en diferentes formatos de codificación. Algunas de estas funciones son:

    tf.strings.unicode_decode: Convierte una cadena de texto codificada (por ejemplo, en UTF-8) en un vector de puntos de código Unicode.

    tf.strings.unicode_encode: Convierte un vector de puntos de código Unicode de vuelta a una cadena codificada.

    tf.strings.unicode_transcode: Convierte una cadena de texto codificada de un formato de codificación a otro (por ejemplo, de UTF-8 a UTF-16).
    
tf.strings.unicode_decode

La función tf.strings.unicode_decode toma una cadena de texto codificada (por ejemplo, UTF-8) y la convierte en un tensor de puntos de código Unicode.

In [ ]:
# Cadena codificada en UTF-8
text_utf8 = tf.constant("Ciencia de datos")

# Decodificar la cadena UTF-8 a puntos de código Unicode
decoded_text = tf.strings.unicode_decode(text_utf8, input_encoding='UTF-8')

# Mostrar el resultado
print(decoded_text)

In [ ]:
tf.strings.unicode_transcode(text_utf8,
                             input_encoding='UTF8',
                             output_encoding='UTF-16-BE')

Convertir de puntos de código Unicode a texto:

In [ ]:
text_chars = tf.constant([[104, 111, 108, 97]])  # 'Hola' en código Unicode
encoded_text = tf.strings.unicode_encode(text_chars, output_encoding='UTF-8')
encoded_text

Transcodificar una cadena a otro tipo de codificación:

In [ ]:
transcoded_text = tf.strings.unicode_transcode(text_utf8, input_encoding='UTF-8', output_encoding='UTF-16-BE')
transcoded_text

Manejo de lotes de cadenas

Decodificar un lote de cadenas:


In [ ]:
batch_utf8 = [s.encode('UTF-8') for s in ['Hola', 'Cuál es el módulo', '😊']]
batch_chars_ragged = tf.strings.unicode_decode(batch_utf8, input_encoding='UTF-8')
for sentence_chars in batch_chars_ragged.to_list():
  print(sentence_chars)

Convertir un RaggedTensor a un tensor denso o disperso:

In [ ]:
batch_chars_padded = batch_chars_ragged.to_tensor(default_value=-1)
batch_chars_sparse = batch_chars_ragged.to_sparse()

batch_chars_padded

Operaciones de Longitud de Caracteres

Calcular el número de bytes y caracteres en una cadena UTF-8:

In [ ]:
thanks = u'Maestría 😊'.encode('UTF-8')
num_bytes = tf.strings.length(thanks).numpy()  # Longitud en bytes
num_chars = tf.strings.length(thanks, unit='UTF8_CHAR').numpy()  # Longitud en caracteres UTF-8
num_bytes,num_chars

Subcadenas y División de Texto

Obtener una subcadena de un texto en formato UTF-8:

In [ ]:
substring = tf.strings.substr(thanks, pos=7, len=1, unit='UTF8_CHAR')
substring

Dividir una cadena Unicode en caracteres individuales:

In [ ]:
split_text = tf.strings.unicode_split(thanks, 'UTF-8')
split_text

---

# 🔧 3. Pipeline Clásico de Clasificación de Texto con Keras

---

## El `Tokenizer` de Keras: herramienta legacy pero fundamental

El `Tokenizer` de `keras.preprocessing.text` es la forma clásica (pre-TF2.4) de convertir texto en secuencias numéricas. Aunque hoy se recomienda `TextVectorization` (visto en el Cuaderno 11), el `Tokenizer` sigue siendo muy utilizado en código existente y tutoriales.

### ¿Qué hace el Tokenizer?

```
Texto: "el gato come pescado"
         ↓ fit_on_texts()
Vocabulario: {"el":1, "gato":2, "come":3, "pescado":4, ...}
         ↓ texts_to_sequences()
Secuencia: [1, 2, 3, 4]
         ↓ pad_sequences()
Padded:  [0, 0, 1, 2, 3, 4]   ← padding hasta max_len
```

### Parámetros clave

| Parámetro | Descripción | Valor típico |
|---|---|---|
| `num_words` | Tamaño máximo del vocabulario | 5000–50000 |
| `oov_token` | Token para palabras desconocidas | `"<OOV>"` |
| `lower` | Convertir a minúsculas | `True` |
| `filters` | Caracteres a eliminar | `'!"#$%&...'` |

### Diferencia con `TextVectorization`

| | `Tokenizer` (legacy) | `TextVectorization` (moderno) |
|--|--|--|
| Integración con grafo TF | ❌ Solo Python | ✅ Capa Keras |
| Exportable con el modelo | ❌ Separado | ✅ Parte del modelo |
| Inferencia en producción | Requiere guardar por separado | ✅ Automático |
| Flexibilidad | Alta | Media-alta |

## Implementación de un clasificador de texto simple con TensorFlow


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, SpatialDropout1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


## Definir los datos de entrenamiento

> ⚠️ **Nota pedagógica**: Este ejemplo usa apenas 3 frases para ilustrar el pipeline técnico paso a paso. Con tan pocos datos, el modelo no puede generalizar. En la sección siguiente construiremos un clasificador con un dataset más representativo.

Los textos tienen etiquetas binarias: **1 = positivo**, **0 = negativo**.


In [ ]:
# Datos de ejemplo
texts = ["Amo la maestría en ciencia de datos", "El lenguaje C es difícil de aprender", "La universidad es lo mejor"]
labels = [1, 0, 1]  # 1: positivo, 0: negativo

## Preprocesamiento de texto

El preprocesamiento incluye convertir los textos en secuencias de enteros y luego hacer padding para asegurar que todas las secuencias tengan la misma longitud.

In [ ]:
# Preprocesamiento de texto
tokenizer = Tokenizer(num_words=50, split=' ')
tokenizer.fit_on_texts(texts)
X = tokenizer.texts_to_sequences(texts)
X = pad_sequences(X)


## Crear el modelo

Aquí definimos un modelo secuencial con una capa de embeddings, una capa LSTM y una capa densa de salida. Este es un modelo muy simple, adecuado para un conjunto de datos pequeño como el de este ejemplo.

In [ ]:
# Modelo de clasificación simple
model = Sequential()
model.add(Embedding(input_dim=50, output_dim=50, input_length=X.shape[1]))
model.add(SpatialDropout1D(0.2))
model.add(LSTM(100, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()



## Entrenamiento del modelo

En esta etapa convertimos los datos y etiquetas a tensores de TensorFlow y entrenamos el modelo con los datos de entrenamiento.

In [ ]:
# Entrenamiento
# Convert X and labels to TensorFlow tensors
X = tf.convert_to_tensor(X)
labels = tf.convert_to_tensor(labels)

model.fit(X, labels, epochs=5, batch_size=2)

## Realizar predicciones con nuevos textos

Una vez entrenado el modelo, podemos usarlo para predecir las etiquetas de nuevos textos. Los nuevos textos deben pasar por el mismo proceso de preprocesamiento antes de ser introducidos en el modelo.

In [ ]:
# Nuevos textos para predecir
new_texts = ["Me encanta aprender sobre IA", "Este examen es dificil"]

# Preprocesar los nuevos textos
new_X = tokenizer.texts_to_sequences(new_texts)
new_X = pad_sequences(new_X, maxlen=X.shape[1])  # Debe tener el mismo largo que X

# Hacer predicciones con el modelo
predictions = model.predict(new_X)

# Mostrar las predicciones
for i, text in enumerate(new_texts):
    print(f"Texto: {text} -> Predicción: {'Positivo' if predictions[i] > 0.5 else 'Negativo'}")

---

# 🔬 6. RNN desde Cero con NumPy — Diseccionando la Recurrencia

---

## ¿Por qué construir una RNN desde cero?

Keras y TensorFlow hacen que entrenar una RNN sea tan fácil como escribir 5 líneas de código. Pero esa facilidad **oculta** la matemática real que ocurre dentro. Construir una RNN con NumPy puro te obliga a entender:

- Cómo fluye la información hacia adelante (**forward pass**).
- Cómo se calculan los gradientes hacia atrás (**BPTT**).
- Por qué el estado oculto es la "memoria" de la red.
- Por qué aparece el problema del gradiente desvanecido.

## La RNN mínima en NumPy

Una RNN con una sola capa oculta tiene solo **tres matrices de pesos**:

| Peso | Forma | Función |
|------|-------|---------|
| $W_{xh}$ | `(hidden_dim, input_dim)` | Transforma la entrada $x_t$ al espacio oculto |
| $W_{hh}$ | `(hidden_dim, hidden_dim)` | Propaga el estado oculto $h_{t-1}$ |
| $W_{hy}$ | `(output_dim, hidden_dim)` | Proyecta el estado oculto a la salida |

La actualización en cada paso de tiempo es:

$$h_t = \sigma(W_{xh} \cdot x_t + W_{hh} \cdot h_{t-1} + b_h)$$
$$\hat{y} = \text{softmax}(W_{hy} \cdot h_t + b_y)$$

> Esta RNN es equivalente a `SimpleRNN` en Keras, pero escrita explícitamente.

## El corpus: nombres de municipios de Colombia

Usaremos una lista de municipios colombianos para entrenar la red. La tarea: dado un carácter, predecir el siguiente. Así la red aprenderá patrones fonéticos del español colombiano.

Esta es una libreta para generar nombres de municipios usando una red recurrente hecha solo con NumPy.


In [ ]:
!wget https://gist.githubusercontent.com/abuiles/306259/raw/932652106fc7c2a2e09f159f4bcabf4a12313524/ciudades.txt

## Apertura del archivo

En esta celda, cargamos el archivo que contiene los nombres de las ciudades. Este archivo se usará para entrenar nuestra red neuronal recurrente. Para este proyecto, asumimos que el archivo contiene tres columnas: el nombre del departamento, el código del municipio y el nombre de la ciudad, separados por tabulaciones o espacios. Extraeremos únicamente los nombres de las ciudades para preparar los datos.

En esta celda, cargamos el archivo ciudades.txt, que contiene tres columnas: nombre del departamento, código del municipio y nombre de la ciudad. Utilizamos el método split('\t') para dividir cada línea del archivo por tabulaciones y extraemos únicamente el nombre de la ciudad, que está en la tercera columna. Luego, se elimina cualquier espacio extra alrededor del nombre de la ciudad utilizando el método strip().

Finalmente, mostramos los primeros cinco nombres de ciudades de la lista para asegurarnos de que la carga y el procesamiento se realizaron correctamente.


In [ ]:
## Importamos lo necesario que es solo numpy
import numpy as np

with open('ciudades.txt', 'r', encoding='utf-8') as file:
    # Leer las líneas del archivo y extraer solo el nombre de la ciudad (última columna)
    nombres_ciudades = [line.split('\t')[2].strip() for line in file.readlines()]

# Mostrar los primeros 5 nombres de ciudades para verificar que la carga fue correcta
nombres_ciudades[:5]

Preprocesamiento de los nombres de ciudades

En esta celda, vamos a procesar los nombres de las ciudades para prepararlos como datos de entrada para nuestra red neuronal recurrente. Esto incluye la creación de un vocabulario de caracteres, la conversión de los nombres a secuencias de enteros y la organización de los datos para el entrenamiento de la RNN.

En esta celda, se genera un vocabulario único de caracteres presentes en los nombres de ciudades. Posteriormente, cada nombre es convertido en una secuencia de índices, donde cada carácter es reemplazado por su correspondiente índice en el vocabulario.

Luego, creamos las secuencias de entrada y salida que alimentarán la red neuronal. Cada entrada es un carácter de un nombre, y la salida es el carácter siguiente en el nombre. Estas secuencias serán utilizadas para entrenar la red neuronal.

El conjunto de datos procesado es convertido en arrays de Numpy, lo que facilita su manejo durante el entrenamiento.

In [ ]:
import numpy as np

# Crear un conjunto único de caracteres presentes en los nombres de ciudades
vocabulario = sorted(set(''.join(nombres_ciudades)))
caracter_a_indice = {caracter: indice for indice, caracter in enumerate(vocabulario)}
indice_a_caracter = {indice: caracter for caracter, indice in caracter_a_indice.items()}

# Convertir los nombres a secuencias de índices
nombres_secuencias = [[caracter_a_indice[caracter] for caracter in nombre] for nombre in nombres_ciudades]

# Definir las secuencias de entrada y salida para la red
sec_entrada = []
sec_salida = []

# Crear las secuencias de entrada y salida (por ejemplo, el primer carácter de un nombre es la entrada, el segundo es la salida)
for nombre in nombres_secuencias:
    for i in range(len(nombre) - 1):
        sec_entrada.append(nombre[i])
        sec_salida.append(nombre[i + 1])

# Convertir las secuencias a arrays de Numpy
X = np.array(sec_entrada)
y = np.array(sec_salida)

# Mostrar información sobre los datos procesados
len(vocabulario), len(X), len(y), vocabulario


## Implementación de la RNN con NumPy

### Arquitectura

```
Entrada x_t (one-hot, tamaño = vocab)
         │
         ▼  Wxh
Estado oculto h_t = σ(Wxh·x_t + Whh·h_{t-1} + bh)   ← RECURRENCIA
         │
         ▼  Why
Salida ŷ = softmax(Why·h_t + by)
         │
         ▼
Probabilidad del siguiente carácter
```

### Inicialización de pesos

Los pesos se inicializan con valores pequeños aleatorios (`* 0.01`) para evitar que la activación sigmoide sature desde el principio. Los sesgos se inicializan a cero.


In [ ]:
# Definir parámetros de la RNN
input_dim = len(vocabulario)  # Número de caracteres en el vocabulario
output_dim = len(vocabulario)  # La salida será un vector de probabilidad para cada carácter
hidden_dim = 128  # Tamaño de la capa oculta (número de unidades LSTM)

# Inicialización de los pesos y sesgos de la red
Wxh = np.random.randn(hidden_dim, input_dim) * 0.01  # Pesos de entrada a capa oculta
Whh = np.random.randn(hidden_dim, hidden_dim) * 0.01  # Pesos de capa oculta a capa oculta
Why = np.random.randn(output_dim, hidden_dim) * 0.01  # Pesos de capa oculta a salida

bh = np.zeros((hidden_dim, 1))  # Sesgo para la capa oculta
by = np.zeros((output_dim, 1))  # Sesgo para la salida

# Función de activación sigmoide
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Función softmax para la capa de salida
def softmax(x):
    e_x = np.exp(x - np.max(x))  # Para estabilidad numérica
    return e_x / e_x.sum(axis=0, keepdims=True)

# Propagación hacia adelante
def forward(X):
    h = np.zeros((hidden_dim, 1))  # Estado oculto inicializado a ceros
    for t in range(len(X)):
        x_t = np.zeros((input_dim, 1))  # Vector de entrada en t
        x_t[X[t]] = 1  # Codificar la entrada como un vector one-hot
        h = sigmoid(np.dot(Wxh, x_t) + np.dot(Whh, h) + bh)  # Capa oculta
    y = np.dot(Why, h) + by  # Capa de salida
    return softmax(y), h  # Probabilidades para el siguiente carácter, estado oculto

# Visualizar el modelo con un paso de prueba
sample_input = X[:1]  # Tomamos el primer ejemplo para la predicción
output, hidden_state = forward(sample_input)
output.argmax(axis=0)


## Entrenamiento: Forward Pass y Pérdida

### Función de pérdida: entropía cruzada

Para clasificación de caracteres usamos **entropía cruzada**:

$$L = -\log(\hat{y}_{y_{true}})$$

Donde $\hat{y}_{y_{true}}$ es la probabilidad asignada al carácter correcto. Cuanto mayor sea esa probabilidad, menor es la pérdida.

### Nota sobre el Backward Pass (BPTT)

En esta implementación simplificada el backward pass (BPTT) no está completamente implementado — los pesos **no se actualizan** durante el bucle de entrenamiento. Esto se hace a propósito para mostrar solo el forward pass con claridad.

En la sección de comparación veremos cómo Keras hace esto automáticamente con `model.fit()`.

> 💡 **Reflexión**: Esta limitación ilustra por qué frameworks como Keras/TensorFlow son tan valiosos — implementar BPTT correctamente requiere calcular gradientes a través de múltiples pasos de tiempo, gestionar la acumulación de gradientes y aplicar actualizaciones numéricamente estables.


In [ ]:
# Función de pérdida (entropía cruzada)
def loss(y_pred, y_true):
    return -np.log(y_pred[y_true])

# Entrenamiento de la red
learning_rate = 0.1
epochs = 1000  # Número de iteraciones

for epoch in range(epochs):
    total_loss = 0  # Inicializar la pérdida total para cada época

    # Iterar sobre cada ejemplo en el conjunto de datos
    for i in range(len(X)):
        # Propagación hacia adelante para un solo carácter
        y_pred, hidden_state = forward([X[i]])  # Pasar un solo carácter como entrada

        # Calcular la pérdida para el carácter actual
        total_loss += loss(y_pred, y[i])  # y[i] es el valor verdadero para el carácter actual

    # Mostrar la pérdida en cada 100 épocas
    if epoch % 100 == 0:
        print(f"Epoch {epoch}/{epochs}, Pérdida: {total_loss}")

    # Aquí deberíamos implementar la retropropagación para actualizar los pesos
    # La retropropagación es una tarea compleja, pero en este ejemplo la omitimos por simplicidad

# Nota: La retropropagación debe actualizar los pesos Wxh, Whh, Why, bh, by para minimizar la pérdida

## Generación de nuevos nombres

Una vez que el modelo está entrenado, podemos usarlo para generar nuevos nombres de ciudades.

En esta celda, generamos nuevos nombres de ciudades a partir de un nombre semilla. La red usa la semilla para predecir el siguiente carácter, luego ese carácter se añade al nombre y se utiliza como entrada para predecir el siguiente carácter, repitiendo el proceso.

Se genera un nuevo nombre con una longitud especificada (por ejemplo, 6 caracteres) utilizando la red entrenada.

In [ ]:
# Función para generar nuevos nombres
def generar_nombre(seed, length=6):
    nombre_generado = seed
    seed_idx = [caracter_a_indice[char] for char in seed]
    for _ in range(length):
        pred, hidden_state = forward(seed_idx)
        next_char_idx = pred.argmax(axis=0)
        # Convert next_char_idx to an integer before using it as a key
        next_char_idx = int(next_char_idx[0]) #Access the element and convert it to int.
        next_char = indice_a_caracter[next_char_idx]
        nombre_generado += next_char
        seed_idx = seed_idx[1:] + [next_char_idx]  # Desplazar la secuencia
    return nombre_generado

# Generar un nombre de ciudad a partir de una semilla
seed = 'An'  # Semilla inicial (por ejemplo, "An")
nuevo_nombre = generar_nombre(seed)
print(f"Nombre generado: {nuevo_nombre}")

---

# 📊 4. Clasificador de Sentimientos Ampliado

En esta sección construimos un clasificador real con un dataset más representativo, métricas de evaluación completas y curvas de aprendizaje. Este es el puente entre el ejemplo pedagógico simple y un sistema de producción.

## Dataset ampliado en español

Usamos 20 frases con balance entre positivas y negativas, incorporando vocabulario variado para que el modelo deba generalizar.


In [ ]:
# ─────────────────────────────────────────────────────
# CLASIFICADOR DE SENTIMIENTOS AMPLIADO
# Pipeline: Tokenizer → pad_sequences → Embedding → LSTM → Dense
# ─────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

# ── Dataset ampliado ──
textos = [
    # Positivos (1)
    "amo la maestría en ciencia de datos es increíble",
    "la universidad ofrece clases excelentes y profesores geniales",
    "me encanta aprender sobre inteligencia artificial cada día",
    "el curso de machine learning es fantástico muy recomendado",
    "estoy muy satisfecho con mi carrera académica y mis logros",
    "aprender python y tensorflow es maravilloso para el futuro",
    "los proyectos de datos me apasionan y me motivan siempre",
    "el equipo docente es excepcional y siempre dispuesto a ayudar",
    "este programa académico ha superado todas mis expectativas",
    "la ciencia de datos abre puertas increíbles en el mercado laboral",
    # Negativos (0)
    "el examen fue muy difícil y no pude terminarlo a tiempo",
    "este lenguaje de programación es confuso y frustrante",
    "no entiendo la tarea y el profesor no explica bien los temas",
    "el curso es aburrido y no aprendo nada nuevo en las clases",
    "estoy frustrado con el rendimiento de mi computador lento",
    "la documentación de esta librería es terrible e incompleta",
    "perdí el examen y estoy muy decepcionado con mis resultados",
    "el ambiente de trabajo es estresante y poco colaborativo",
    "no me gustó el módulo de estadística es demasiado complejo",
    "los errores en el código me tienen desesperado hace días",
]
etiquetas = [1]*10 + [0]*10

# ── Hiperparámetros ──
VOCAB_SIZE  = 300
MAX_LEN     = 15
EMBED_DIM   = 32
OOV_TOKEN   = "<OOV>"

# ── Tokenización ──
tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token=OOV_TOKEN)
tokenizer.fit_on_texts(textos)

print(f"Vocabulario aprendido: {len(tokenizer.word_index)} tokens únicos")
print(f"Tokens más frecuentes: {list(tokenizer.word_index.items())[:10]}")

X = tokenizer.texts_to_sequences(textos)
X = pad_sequences(X, maxlen=MAX_LEN, padding='post', truncating='post')
y = np.array(etiquetas)

print(f"\nForma de X: {X.shape}")
print(f"Ejemplo - texto: '{textos[0][:40]}...'")
print(f"          vector: {X[0]}")


In [ ]:
# ─────────────────────────────────────────────────────
# MODELO: Embedding + BiLSTM + Dropout + Dense
# ─────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X_tr, X_ts, y_tr, y_ts = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_sent = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMBED_DIM,
              input_length=MAX_LEN, mask_zero=True, name='embedding'),
    Bidirectional(LSTM(64, dropout=0.2), name='bilstm'),
    Dense(32, activation='relu', name='dense_oculto'),
    Dropout(0.3),
    Dense(1, activation='sigmoid', name='salida')
], name='Clasificador_Sentimientos')

model_sent.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

model_sent.summary()


In [ ]:
# ─────────────────────────────────────────────────────
# ENTRENAR Y VISUALIZAR CURVAS DE APRENDIZAJE
# ─────────────────────────────────────────────────────
history_sent = model_sent.fit(
    X_tr, y_tr,
    epochs=50, batch_size=4,
    validation_data=(X_ts, y_ts),
    verbose=0
)

# ── Métricas finales ──
resultados = model_sent.evaluate(X_ts, y_ts, verbose=0)
print("=== Evaluación en conjunto de prueba ===")
for nombre, valor in zip(model_sent.metrics_names, resultados):
    print(f"  {nombre:12s}: {valor:.4f}")

# ── Curvas de aprendizaje ──
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_sent.history['loss'], color='steelblue', label='Train')
axes[0].plot(history_sent.history['val_loss'], color='tomato', linestyle='--', label='Val')
axes[0].set_title('Pérdida (Binary Crossentropy)'); axes[0].set_xlabel('Época')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history_sent.history['accuracy'], color='steelblue', label='Train Acc')
axes[1].plot(history_sent.history['val_accuracy'], color='tomato', linestyle='--', label='Val Acc')
axes[1].plot(history_sent.history['val_precision'], color='seagreen', linestyle=':', label='Val Precision')
axes[1].plot(history_sent.history['val_recall'], color='orange', linestyle=':', label='Val Recall')
axes[1].set_title('Métricas de clasificación'); axes[1].set_xlabel('Época')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Clasificador de Sentimientos — Entrenamiento', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────
# PREDICCIONES Y ANÁLISIS DE ERRORES
# ─────────────────────────────────────────────────────
frases_nuevas = [
    "me encanta trabajar con datos y modelos de IA",
    "este problema de matemáticas es imposible de entender",
    "el modelo de deep learning funciona perfectamente",
    "estoy agotado de depurar este código interminable",
    "aprender es un proceso emocionante y enriquecedor",
]

X_nuevo = tokenizer.texts_to_sequences(frases_nuevas)
X_nuevo = pad_sequences(X_nuevo, maxlen=MAX_LEN, padding='post')
preds = model_sent.predict(X_nuevo, verbose=0)

print("=== Predicciones ===\n")
for frase, pred in zip(frases_nuevas, preds):
    label = "✅ POSITIVO" if pred[0] > 0.5 else "❌ NEGATIVO"
    confianza = pred[0] if pred[0] > 0.5 else 1 - pred[0]
    print(f"  Texto: '{frase[:50]}'")
    print(f"  → {label}  (confianza: {confianza:.1%})\n")

# ── Análisis de la capa Embedding ──
print("\n=== Tokens más importantes aprendidos ===")
pesos_embed = model_sent.get_layer('embedding').get_weights()[0]
vocab_inv = {v: k for k, v in tokenizer.word_index.items()}
# Palabras con mayor norma en el espacio de embedding = más informativas
normas = np.linalg.norm(pesos_embed, axis=1)
top_idx = np.argsort(normas)[::-1][1:16]  # excluir idx 0 (padding)
print("  Palabras con mayor norma en el embedding (más informativas):")
for idx in top_idx:
    word = vocab_inv.get(idx, f'<idx_{idx}>')
    print(f"    {word:20s} norma={normas[idx]:.4f}")


---

# ⚖️ 5. Comparativa: `Tokenizer` (legacy) vs `TextVectorization` (moderno)

Esta sección ilustra las diferencias prácticas entre las dos herramientas de vectorización de texto en el ecosistema Keras.

## El problema del `Tokenizer` en producción

Cuando guardas un modelo de Keras entrenado con `Tokenizer`, **el tokenizador no se guarda junto con el modelo**. Debes serializar y guardar el objeto por separado (con `pickle` o JSON), y cargar y sincronizar versiones en inferencia.

Con `TextVectorization`, la capa se guarda como parte del modelo. No hay desincronización posible.


In [ ]:
# ─────────────────────────────────────────────────────
# COMPARATIVA: Tokenizer legacy vs TextVectorization moderno
# ─────────────────────────────────────────────────────
import tensorflow as tf
import numpy as np

corpus_demo = [
    "el procesamiento de lenguaje natural es fascinante",
    "las redes neuronales aprenden de los datos del texto",
    "los embeddings representan palabras como vectores densos",
    "la inteligencia artificial transforma el mundo actual",
]

VOCAB = 100
MAX_L = 10

# ── Opción A: Tokenizer legacy ──
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

tok_legacy = Tokenizer(num_words=VOCAB, oov_token='<OOV>')
tok_legacy.fit_on_texts(corpus_demo)
seq_legacy = tok_legacy.texts_to_sequences(corpus_demo)
X_legacy   = pad_sequences(seq_legacy, maxlen=MAX_L, padding='post')

# ── Opción B: TextVectorization moderno ──
tv = tf.keras.layers.TextVectorization(max_tokens=VOCAB, output_sequence_length=MAX_L)
tv.adapt(corpus_demo)
X_modern = tv(corpus_demo).numpy()

# ── Comparar salidas ──
print("=== Tokenizer legacy ===")
print(f"Vocabulario: {len(tok_legacy.word_index)} tokens")
print(f"Primera frase: {X_legacy[0]}")

print("\n=== TextVectorization moderno ===")
vocab_tv = tv.get_vocabulary()
print(f"Vocabulario: {len(vocab_tv)} tokens")
print(f"Primera frase: {X_modern[0]}")

print("\n=== Diferencias clave ===")
diferencias = [
    ("¿Se guarda con el modelo?",      "❌ No (serializar aparte)", "✅ Sí (capa Keras)"),
    ("¿Funciona en TF graph?",         "❌ Solo Python",            "✅ Sí"),
    ("¿Soporte batch automático?",     "Manual",                   "✅ Automático"),
    ("¿Token OOV configurable?",       "✅ Sí (oov_token)",        "✅ Sí ([UNK])"),
    ("¿Normalización integrada?",      "Parcial",                  "✅ lowercase, strip_punctuation"),
    ("¿Recomendado para producción?",  "❌ No",                    "✅ Sí"),
]

print(f"{'Característica':<35} {'Tokenizer':^25} {'TextVectorization':^25}")
print("-" * 87)
for feat, tok, tv_val in diferencias:
    print(f"{feat:<35} {tok:^25} {tv_val:^25}")


---

# ⚗️ 7. Comparación: RNN NumPy vs RNN Keras

Esta sección conecta la implementación manual (Sección 6) con la equivalencia en Keras.

## Equivalencia matemática

| NumPy (manual) | Keras equivalente |
|---|---|
| `Wxh`, `Whh`, `bh` | Pesos internos de `SimpleRNN` |
| `Why`, `by` | Pesos de `Dense` |
| `sigmoid(Wxh·x + Whh·h + bh)` | `SimpleRNN(activation='sigmoid')` |
| `softmax(Why·h + by)` | `Dense(vocab_size, activation='softmax')` |
| Bucle `for t in range(T)` | Bucle interno de `SimpleRNN` (automático) |
| Cálculo manual de gradientes | `model.fit()` + autodiferenciación |

## Ventajas de cada enfoque

| | NumPy manual | Keras |
|--|--|--|
| **Comprensión** | ✅ Visibilidad total | ❌ Opaco internamente |
| **Velocidad** | ❌ Lento (Python puro) | ✅ Rápido (C++, GPU) |
| **Escalabilidad** | ❌ No escala | ✅ Millones de parámetros |
| **BPTT automático** | ❌ Manual | ✅ Autodiferenciación |
| **Objetivo** | Aprendizaje | Producción |


In [ ]:
# ─────────────────────────────────────────────────────
# EQUIVALENTE EN KERAS de la RNN NumPy manual
# ─────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import requests, re, unicodedata

tf.random.set_seed(42); np.random.seed(42)

# ── Cargar corpus de municipios ──
url = "https://gist.githubusercontent.com/abuiles/306259/raw/932652106fc7c2a2e09f159f4bcabf4a12313524/ciudades.txt"
try:
    resp = requests.get(url, timeout=10)
    lineas = resp.text.strip().splitlines()
    municipios = []
    for linea in lineas:
        partes = linea.split('\t')
        if len(partes) >= 3:
            nombre = partes[2].strip().lower()
            nombre = unicodedata.normalize('NFKD', nombre)
            nombre = ''.join(c for c in nombre if not unicodedata.combining(c))
            nombre = re.sub(r'[^a-z\s]', '', nombre).strip()
            if nombre:
                municipios.append(nombre)
    print(f"Municipios cargados: {len(municipios)}")
    print(f"Ejemplos: {municipios[:5]}")
except:
    # fallback si no hay conexión
    municipios = ["bogota", "medellin", "cali", "barranquilla", "cartagena",
                  "cucuta", "bucaramanga", "pereira", "manizales", "armenia",
                  "villavicencio", "santa marta", "neiva", "pasto", "ibague"]
    print(f"Usando fallback local. Municipios: {len(municipios)}")

# ── Preparar vocabulario y secuencias ──
texto_corpus = '\n'.join(municipios) + '\n'
chars = sorted(set(texto_corpus))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}
VOCAB_SIZE_M = len(chars)

SEQLEN = 12
X_m, y_m = [], []
for i in range(len(texto_corpus) - SEQLEN):
    X_m.append([char2idx[c] for c in texto_corpus[i:i+SEQLEN]])
    y_m.append(char2idx[texto_corpus[i+SEQLEN]])

X_m = np.array(X_m); y_m = np.array(y_m)
print(f"\nSecuencias de entrenamiento: {X_m.shape}")
print(f"Vocabulario: {VOCAB_SIZE_M} caracteres → {chars}")


In [ ]:
# ─────────────────────────────────────────────────────
# MODELOS EN KERAS: SimpleRNN vs LSTM vs GRU
# para generación de nombres de municipios
# ─────────────────────────────────────────────────────
EMBED_M  = 32
HIDDEN_M = 128

def crear_modelo_municipios(tipo):
    rnn_layer = {
        'SimpleRNN': layers.SimpleRNN(HIDDEN_M, dropout=0.15),
        'LSTM':      layers.LSTM(HIDDEN_M, dropout=0.15),
        'GRU':       layers.GRU(HIDDEN_M, dropout=0.15),
    }[tipo]
    m = models.Sequential([
        layers.Embedding(VOCAB_SIZE_M, EMBED_M, name='embedding'),
        rnn_layer,
        layers.Dense(VOCAB_SIZE_M, activation='softmax', name='salida')
    ], name=f'Municipios_{tipo}')
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
    return m

# Entrenar los tres modelos
histories_m = {}
modelos_m   = {}

for tipo in ['SimpleRNN', 'LSTM', 'GRU']:
    print(f"\nEntrenando {tipo}...")
    m = crear_modelo_municipios(tipo)
    h = m.fit(X_m, y_m, epochs=20, batch_size=128,
              validation_split=0.1, verbose=0)
    histories_m[tipo] = h.history
    modelos_m[tipo]   = m
    print(f"  Loss final: {h.history['loss'][-1]:.4f} | Acc: {h.history['accuracy'][-1]:.2%}")

# ── Curvas comparativas ──
import matplotlib.pyplot as plt
colores = {'SimpleRNN': 'tomato', 'LSTM': 'steelblue', 'GRU': 'seagreen'}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
for tipo, hist in histories_m.items():
    ax1.plot(hist['val_loss'],     color=colores[tipo], label=tipo)
    ax2.plot(hist['val_accuracy'], color=colores[tipo], label=tipo)

ax1.set_title('Val Loss — Generación de Municipios'); ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_title('Val Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)
plt.suptitle('Comparación SimpleRNN vs LSTM vs GRU en generación de municipios', fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────
# GENERAR NOMBRES DE MUNICIPIOS CON TEMPERATURA
# ─────────────────────────────────────────────────────
import numpy as np

def sample_temp(preds, temperatura=0.7):
    preds = np.log(np.asarray(preds, dtype='float64') + 1e-8) / temperatura
    exp_p = np.exp(preds)
    preds = exp_p / exp_p.sum()
    return np.random.choice(len(preds), p=preds)

def generar_municipio(modelo, seed, max_len=20, temperatura=0.7):
    seed = seed.lower()[:SEQLEN].ljust(SEQLEN)
    resultado = seed.strip()
    for _ in range(max_len):
        x_in = np.array([[char2idx.get(c, 0) for c in seed]])
        pred = modelo.predict(x_in, verbose=0)[0]
        next_idx = sample_temp(pred, temperatura)
        next_c = idx2char[next_idx]
        if next_c == '\n':
            break
        resultado += next_c
        seed = seed[1:] + next_c
    return resultado.strip()

print("=== Nombres de municipios generados ===\n")
for tipo, modelo in modelos_m.items():
    print(f"── {tipo} (temperatura=0.7) ──")
    nombres = [generar_municipio(modelo, seed='an', temperatura=0.7) for _ in range(5)]
    for n in nombres:
        print(f"  {n}")
    print()


---

# 🏋️ 8. Ejercicios Propuestos

---

**Ejercicio 1 — Unicode y puntos de código**
Escribe una función que reciba cualquier cadena de texto en español y devuelva:
- El número de caracteres Unicode.
- El número de bytes en UTF-8.
- Los 5 caracteres con mayor punto de código (los más "exóticos").

**Ejercicio 2 — Tokenizer vs TextVectorization**
Vectoriza el mismo corpus con `Tokenizer` y con `TextVectorization`. ¿Los índices asignados a las mismas palabras son iguales? ¿Por qué sí o por qué no?

**Ejercicio 3 — Ampliar el clasificador de sentimientos**
Agrega 10 frases más al dataset (5 positivas, 5 negativas) relacionadas con el ámbito universitario. ¿Mejora el accuracy de validación? ¿Cuántas épocas de entrenamiento son necesarias?

**Ejercicio 4 — Backpropagation en la RNN NumPy**
La implementación de la RNN NumPy omite el backward pass. Implementa la actualización de **al menos el peso `Why`**:
```python
# Gradiente de la salida softmax con respecto a Why
dL_dWhy = np.outer(dL_dy, h.T)
Why -= learning_rate * dL_dWhy
```
¿Disminuye la pérdida ahora?

**Ejercicio 5 — OOV tokens**
Entrena el `Tokenizer` con el corpus de entrenamiento. Luego predice con frases que contengan palabras **no vistas** durante el entrenamiento. ¿Cómo maneja el modelo las palabras OOV? Compara el comportamiento con `oov_token=None` vs `oov_token='<OOV>'`.

**Ejercicio 6 — Temperatura en generación**
Con el modelo de municipios entrenado, genera 10 nombres con temperatura `0.2` y 10 con temperatura `1.5`. Clasifica cuántos "suenan" como municipios reales colombianos. ¿A qué temperatura el balance creatividad/coherencia es mejor?

**Ejercicio 7 — Integración con el Cuaderno 11**
Usa el modelo de municipios del Cuaderno 11 (FastText) para:
1. Obtener el embedding de la palabra "bogota".
2. Encontrar los 5 municipios más similares usando similitud coseno.
3. Compara estos resultados con los nombres generados por la RNN de este cuaderno.

**Ejercicio 8 — SpatialDropout1D**
El modelo de clasificación de texto usa `SpatialDropout1D`. Investiga:
- ¿En qué se diferencia de `Dropout` normal?
- ¿Por qué es más apropiado para secuencias de embeddings?
- Compara la curva de validación con y sin `SpatialDropout1D`.
